In [2]:
!pip install -q seqeval datasets transformers accelerate packaging

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# -*- coding: utf-8 -*-
# Colab 最終版：
# JSON → BIO（T* 自動比對）→ raw_datasets
# DLR 真實資料先依比例抽樣（50% / 100%）
# 再切分：train/validation/test = 60/20/20
# SYN 合成資料依比例抽樣（50% / 100%）
# synthetic 僅加入 train
# validation 用於 early stopping / best checkpoint selection
# test 僅作最後正式評估
# 單次訓練到 MAX_EPOCHS（每 epoch eval/save）
# 僅保留 1 個最佳 checkpoint（以 eval_f1 決定）
# 輸出 epoch_metrics.csv / classification_report_best.txt / per_class_scores_best.csv / results_summary.csv

import os
import gc
import json
import time
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
import transformers
from packaging import version
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification, TrainingArguments, Trainer,
    EarlyStoppingCallback, set_seed
)
from seqeval.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, classification_report
)

# =========================
# Colab 掛載 Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

# ======== 版本檢查 ========
MIN_VER = "4.26.0"
if version.parse(transformers.__version__) < version.parse(MIN_VER):
    raise RuntimeError(
        f"Your transformers=={transformers.__version__} is too old. "
        f"Please upgrade to >= {MIN_VER}."
    )

# ===================== 可調參數區（Colab 免費版建議） =====================
# 請改成你自己的路徑
INPUT_JSON = "/content/drive/MyDrive/20260321/DLR_cleaned_no_none.json"   # 真實資料 DLR
SYN_JSON   = "/content/drive/MyDrive/20260321/SYN_Similar.json"            # 合成資料 SYN

MODEL_NAME = "bert-base-cased"

MAX_LEN = 128
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01

LOG_STEPS = 20
EARLY_STOP_PATIENCE = 3
SAVE_TOTAL_LIMIT = 1
MAX_EPOCHS = 20

# 這次比例實驗設定
DLR_RATIO_LIST = [0.5, 1.0]   # 真實資料拿 50%、100%
SYN_RATIO_LIST = [0.3, 0.6]   # 合成資料拿 50%、100%
SAMPLE_SEEDS = [11, 22]

OUT_ROOT = "/content/drive/MyDrive/20260321/runs_ratio_exp"
os.makedirs(OUT_ROOT, exist_ok=True)

# ======== device / mixed precision ========
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else ""
USE_FP16 = torch.cuda.is_available() and any(
    k in gpu_name for k in ["t4", "p100", "v100", "a100", "l4"]
)

print(f"transformers version: {transformers.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | fp16={USE_FP16}")
else:
    print("No GPU detected, running on CPU")

# ========== BIO 轉換工具 ==========
def normalize_text_for_search(text: str) -> str:
    return text.replace("&amp;", "&")

def whitespace_tokenize_with_offsets(text: str) -> List[Tuple[str, int, int]]:
    tokens_with_spans = []
    idx = 0
    for part in text.split():
        start = text.find(part, idx)
        if start < 0:
            start = idx
        end = start + len(part)
        tokens_with_spans.append((part, start, end))
        idx = end + 1
    return tokens_with_spans

def spans_from_T_fields(post_content: Dict) -> List[Tuple[int, int, str]]:
    raw_text = post_content.get("text", "")
    text = normalize_text_for_search(raw_text)
    occupied = [False] * len(text)
    spans: List[Tuple[int, int, str]] = []

    pairs: List[Tuple[str, str]] = []
    for k, v in post_content.items():
        if isinstance(k, str) and k.startswith("T") and isinstance(v, dict):
            etxt = v.get("text")
            ecls = v.get("class")
            if isinstance(etxt, str) and isinstance(ecls, str) and etxt and ecls:
                pairs.append((etxt, ecls))

    for etxt, ecls in pairs:
        start_search = 0
        while True:
            pos = text.find(etxt, start_search)
            if pos == -1:
                break
            end = pos + len(etxt)
            if any(occupied[pos:end]):
                start_search = pos + 1
                continue
            spans.append((pos, end, ecls))
            for i in range(pos, end):
                if 0 <= i < len(occupied):
                    occupied[i] = True
            start_search = end

    spans.sort(key=lambda x: x[0])
    return spans

def bio_from_char_spans(text: str, spans: List[Tuple[int, int, str]]) -> Tuple[List[str], List[str]]:
    toks_with_spans = whitespace_tokenize_with_offsets(text)
    tokens, tags = [], []
    for tok, s, e in toks_with_spans:
        label = "O"
        for st, ed, cls in spans:
            if s >= st and e <= ed:
                label = f"B-{cls}" if s == st else f"I-{cls}"
                break
            if (s < ed and e > st):
                label = f"I-{cls}"
                break
        tokens.append(tok)
        tags.append(label)
    return tokens, tags

def data_to_bio_dfs_from_json(json_path: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    seq_rows, tok_rows = [], []
    for post_id, content in data.items():
        raw_text = content.get("text", "")
        text = normalize_text_for_search(raw_text)
        spans = spans_from_T_fields(content)
        tokens, tags = bio_from_char_spans(text, spans)

        seq_rows.append({
            "post_id": post_id,
            "tokens": tokens,
            "ner_tags": tags,
            "text": text
        })

        for t, y in zip(tokens, tags):
            tok_rows.append({
                "post_id": post_id,
                "token": t,
                "BIO_tag": y
            })

    return pd.DataFrame(seq_rows), pd.DataFrame(tok_rows)

# ========== 抽樣工具 ==========
def sample_dataframe_by_ratio(df: pd.DataFrame, ratio: float, seed: int) -> pd.DataFrame:
    if ratio <= 0 or ratio > 1:
        raise ValueError(f"ratio must be in (0,1], got {ratio}")

    if ratio == 1.0:
        return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    n = max(1, int(len(df) * ratio))
    return df.sample(n=n, random_state=seed).reset_index(drop=True)

# ========== 資料切分：train / validation / test ==========
def split_train_valid_test(
    df: pd.DataFrame,
    train_ratio: float = 0.6,
    valid_ratio: float = 0.2,
    test_ratio: float = 0.2,
    seed: int = 42
):
    total = train_ratio + valid_ratio + test_ratio
    if not np.isclose(total, 1.0):
        raise ValueError(f"train+valid+test must sum to 1.0, got {total}")

    rng = np.random.default_rng(seed)
    idx = np.arange(len(df))
    rng.shuffle(idx)

    n = len(df)
    n_train = int(n * train_ratio)
    n_valid = int(n * valid_ratio)

    train_idx = idx[:n_train]
    valid_idx = idx[n_train:n_train + n_valid]
    test_idx = idx[n_train + n_valid:]

    train_df = df.iloc[train_idx].reset_index(drop=True)
    valid_df = df.iloc[valid_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    return train_df, valid_df, test_df

# ========== 準備資料 ==========
df_seq, df_tok = data_to_bio_dfs_from_json(INPUT_JSON)
print(f"Loaded DLR real data: {len(df_seq)} posts")

df_seq_syn, _ = data_to_bio_dfs_from_json(SYN_JSON)
print(f"Loaded SYN data: {len(df_seq_syn)} posts")

# ========== 標籤表：補齊 BIO 成對 ==========
def build_label_list_with_bio(raw_datasets: Dict[str, Dataset]) -> Tuple[List[str], Dict[str, int], Dict[int, str]]:
    bio_tags = set()
    for split in raw_datasets.values():
        for labs in split["ner_tags"]:
            bio_tags.update(labs)

    types = set()
    for tag in bio_tags:
        if tag == "O":
            continue
        if tag.startswith("B-") or tag.startswith("I-"):
            types.add(tag.split("-", 1)[1])
        else:
            types.add(tag)

    label_list = ["O"] + sorted([f"B-{t}" for t in types] + [f"I-{t}" for t in types])
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}
    return label_list, label2id, id2label

# ========== 建立 datasets ==========
def build_datasets_with_ratio(train_seed: int, dlr_ratio: float, syn_ratio: float):
    # 1) 先從 DLR 抽這次實驗要用的比例
    df_seq_dlr_sub = sample_dataframe_by_ratio(df_seq, ratio=dlr_ratio, seed=train_seed)

    # 2) 再把這批 DLR 切成 train / validation / test
    train_real_seq, valid_seq, test_seq = split_train_valid_test(
        df_seq_dlr_sub,
        train_ratio=0.6,
        valid_ratio=0.2,
        test_ratio=0.2,
        seed=train_seed
    )

    # 3) 從 SYN 抽這次要加入 train 的比例
    syn_sample = sample_dataframe_by_ratio(df_seq_syn, ratio=syn_ratio, seed=train_seed)

    # 4) synthetic 只加入 train
    train_seq = pd.concat([train_real_seq, syn_sample], ignore_index=True)
    train_seq = train_seq.sample(frac=1.0, random_state=train_seed).reset_index(drop=True)

    raw_datasets = {
        "train": Dataset.from_pandas(train_seq.drop(columns=["text"])),
        "validation": Dataset.from_pandas(valid_seq.drop(columns=["text"])),
        "test": Dataset.from_pandas(test_seq.drop(columns=["text"])),
    }

    label_list, label2id, id2label = build_label_list_with_bio(raw_datasets)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_and_align_labels(examples):
        tok = tokenizer(
            examples["tokens"],
            is_split_into_words=True,
            truncation=True,
            max_length=MAX_LEN,
        )

        new_labels = []
        for i, labs in enumerate(examples["ner_tags"]):
            word_ids = tok.word_ids(batch_index=i)
            prev_word_id = None
            lab_ids = []

            for w in word_ids:
                if w is None:
                    lab_ids.append(-100)
                elif w != prev_word_id:
                    lab_ids.append(label2id.get(labs[w], label2id["O"]))
                else:
                    lb = labs[w]
                    if lb.startswith("B-"):
                        lb = "I-" + lb.split("-", 1)[1]
                    elif lb != "O" and not (lb.startswith("I-") or lb.startswith("B-")):
                        lb = f"I-{lb}"
                    lab_ids.append(label2id.get(lb, label2id["O"]))
                prev_word_id = w

            new_labels.append(lab_ids)

        tok["labels"] = new_labels
        return tok

    remove_cols_train = [c for c in raw_datasets["train"].column_names if c not in ("tokens", "ner_tags")]
    remove_cols_valid = [c for c in raw_datasets["validation"].column_names if c not in ("tokens", "ner_tags")]
    remove_cols_test  = [c for c in raw_datasets["test"].column_names if c not in ("tokens", "ner_tags")]

    tokenized_train = raw_datasets["train"].map(
        tokenize_and_align_labels, batched=True, remove_columns=remove_cols_train
    )
    tokenized_valid = raw_datasets["validation"].map(
        tokenize_and_align_labels, batched=True, remove_columns=remove_cols_valid
    )
    tokenized_test = raw_datasets["test"].map(
        tokenize_and_align_labels, batched=True, remove_columns=remove_cols_test
    )

    info = {
        "dlr_total_used": len(df_seq_dlr_sub),
        "train_real_size": len(train_real_seq),
        "valid_size": len(valid_seq),
        "test_size": len(test_seq),
        "syn_total_used": len(syn_sample),
        "train_final_size": len(train_seq),
    }

    return (
        raw_datasets,
        tokenized_train,
        tokenized_valid,
        tokenized_test,
        tokenizer,
        label_list,
        label2id,
        id2label,
        info
    )

# ========== 評估指標 ==========
def make_compute_metrics(label_list):
    def compute_metrics(p):
        preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
        preds = np.argmax(preds, axis=2)
        labels = p.label_ids

        true_labels = [
            [label_list[l] for l in label_row if l != -100]
            for label_row in labels
        ]
        true_preds = [
            [label_list[p_i] for p_i, l in zip(pred_row, label_row) if l != -100]
            for pred_row, label_row in zip(preds, labels)
        ]

        return {
            "precision": precision_score(true_labels, true_preds),
            "recall":    recall_score(true_labels, true_preds),
            "f1":        f1_score(true_labels, true_preds),
            "accuracy":  accuracy_score(true_labels, true_preds),
        }
    return compute_metrics

# ========== 單次訓練 ==========
def run_single_training(
    combo_dir: str,
    max_epochs: int,
    tokenized_train,
    tokenized_valid,
    tokenized_test,
    tokenizer,
    label_list,
    label2id,
    id2label,
    seed: int,
):
    os.makedirs(combo_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=combo_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=max_epochs,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        logging_dir=os.path.join(combo_dir, "logs"),
        logging_steps=LOG_STEPS,
        logging_strategy="steps",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=SAVE_TOTAL_LIMIT,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1",
        greater_is_better=True,
        report_to="none",
        seed=seed,
        fp16=USE_FP16,
        bf16=False,
        optim="adamw_torch",
        dataloader_num_workers=2,
        dataloader_pin_memory=True if torch.cuda.is_available() else False,
    )

    model_local = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    )

    data_collator = DataCollatorForTokenClassification(tokenizer)
    compute_metrics = make_compute_metrics(label_list)

    trainer = Trainer(
        model=model_local,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_valid,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
    )

    set_seed(seed)

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    best_checkpoint = getattr(getattr(trainer, "state", None), "best_model_checkpoint", None)

    logs = trainer.state.log_history
    train_rows, eval_rows = [], []

    for e in logs:
        if "epoch" not in e:
            continue

        if "loss" in e and "eval_loss" not in e:
            train_rows.append({
                "epoch": float(e["epoch"]),
                "train_loss": float(e["loss"])
            })

        if "eval_loss" in e:
            eval_rows.append({
                "epoch": float(e["epoch"]),
                "eval_loss": float(e.get("eval_loss", np.nan)),
                "eval_precision": float(e.get("eval_precision", np.nan)),
                "eval_recall": float(e.get("eval_recall", np.nan)),
                "eval_f1": float(e.get("eval_f1", np.nan)),
                "eval_accuracy": float(e.get("eval_accuracy", np.nan)),
            })

    df_train_curve = (
        pd.DataFrame(train_rows).groupby("epoch", as_index=False).last()
        if train_rows else pd.DataFrame(columns=["epoch", "train_loss"])
    )
    df_eval_curve = (
        pd.DataFrame(eval_rows).groupby("epoch", as_index=False).last()
        if eval_rows else pd.DataFrame(columns=["epoch", "eval_loss", "eval_precision", "eval_recall", "eval_f1", "eval_accuracy"])
    )

    df_curve = pd.merge(df_train_curve, df_eval_curve, on="epoch", how="outer").sort_values("epoch").reset_index(drop=True)
    df_curve.to_csv(os.path.join(combo_dir, "epoch_metrics.csv"), index=False, encoding="utf-8-sig")

    train_metrics = trainer.evaluate(eval_dataset=tokenized_train, metric_key_prefix="train")
    valid_metrics = trainer.evaluate(eval_dataset=tokenized_valid, metric_key_prefix="valid")
    test_metrics  = trainer.evaluate(eval_dataset=tokenized_test,  metric_key_prefix="test")

    pred_output = trainer.predict(tokenized_test)
    pred_ids = pred_output.predictions[0] if isinstance(pred_output.predictions, tuple) else pred_output.predictions
    pred_ids = np.argmax(pred_ids, axis=2)
    label_ids = pred_output.label_ids

    true_labels = [[label_list[l] for l in row if l != -100] for row in label_ids]
    pred_labels = [
        [label_list[p_i] for p_i, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(pred_ids, label_ids)
    ]

    rep_dict = classification_report(true_labels, pred_labels, digits=4, output_dict=True)

    with open(os.path.join(combo_dir, "classification_report_best.txt"), "w", encoding="utf-8") as f:
        f.write(classification_report(true_labels, pred_labels, digits=4))

    rows = []
    for k, v in rep_dict.items():
        if isinstance(v, dict) and {"precision", "recall", "f1-score"} <= v.keys():
            if k not in ["micro avg", "macro avg", "weighted avg", "accuracy"]:
                rows.append({
                    "entity": k,
                    "precision": v["precision"],
                    "recall": v["recall"],
                    "f1": v["f1-score"]
                })

    pd.DataFrame(rows).to_csv(
        os.path.join(combo_dir, "per_class_scores_best.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    best_epoch = None
    if "eval_f1" in df_curve.columns and len(df_curve):
        best_epoch = float(df_curve.loc[df_curve["eval_f1"].idxmax(), "epoch"])

    summary = {
        "best_epoch": best_epoch if best_epoch is not None else -1,
        "best_checkpoint": best_checkpoint,
        "train_time_sec": float(train_time),

        "train_loss": float(train_metrics.get("train_loss", np.nan)),
        "train_precision": float(train_metrics.get("train_precision", np.nan)),
        "train_recall": float(train_metrics.get("train_recall", np.nan)),
        "train_f1": float(train_metrics.get("train_f1", np.nan)),
        "train_accuracy": float(train_metrics.get("train_accuracy", np.nan)),

        "valid_loss": float(valid_metrics.get("valid_loss", np.nan)),
        "valid_precision": float(valid_metrics.get("valid_precision", np.nan)),
        "valid_recall": float(valid_metrics.get("valid_recall", np.nan)),
        "valid_f1": float(valid_metrics.get("valid_f1", np.nan)),
        "valid_accuracy": float(valid_metrics.get("valid_accuracy", np.nan)),

        "test_loss": float(test_metrics.get("test_loss", np.nan)),
        "test_precision": float(test_metrics.get("test_precision", np.nan)),
        "test_recall": float(test_metrics.get("test_recall", np.nan)),
        "test_f1": float(test_metrics.get("test_f1", np.nan)),
        "test_accuracy": float(test_metrics.get("test_accuracy", np.nan)),
    }

    del pred_output, pred_ids, label_ids
    del trainer, model_local
    gc.collect()
    torch.cuda.empty_cache()

    return summary

# ========== 主流程：4 種比例組合 × seeds ==========
all_runs = []

for dlr_ratio in DLR_RATIO_LIST:
    for syn_ratio in SYN_RATIO_LIST:
        for seed in SAMPLE_SEEDS:
            print(f"\n===== Building datasets | DLR={dlr_ratio:.0%} | SYN={syn_ratio:.0%} | seed={seed} =====")

            (
                raw_dsets,
                tokenized_train,
                tokenized_valid,
                tokenized_test,
                tokenizer,
                label_list,
                label2id,
                id2label,
                info
            ) = build_datasets_with_ratio(
                train_seed=seed,
                dlr_ratio=dlr_ratio,
                syn_ratio=syn_ratio
            )

            combo_dir = os.path.join(
                OUT_ROOT,
                f"DLR_{int(dlr_ratio*100)}_SYN_{int(syn_ratio*100)}_seed{seed}"
            )
            os.makedirs(combo_dir, exist_ok=True)

            print(f"DLR total used : {info['dlr_total_used']}")
            print(f"Train real size: {info['train_real_size']}")
            print(f"Valid size     : {info['valid_size']}")
            print(f"Test size      : {info['test_size']}")
            print(f"SYN total used : {info['syn_total_used']}")
            print(f"Train final    : {info['train_final_size']}")

            print(f"\n========== Training | DLR={dlr_ratio:.0%} | SYN={syn_ratio:.0%} | seed={seed} ==========")

            summary = run_single_training(
                combo_dir=combo_dir,
                max_epochs=MAX_EPOCHS,
                tokenized_train=tokenized_train,
                tokenized_valid=tokenized_valid,
                tokenized_test=tokenized_test,
                tokenizer=tokenizer,
                label_list=label_list,
                label2id=label2id,
                id2label=id2label,
                seed=seed,
            )

            summary["dlr_ratio"] = dlr_ratio
            summary["syn_ratio"] = syn_ratio
            summary["sample_seed"] = seed
            summary["dlr_total_used"] = info["dlr_total_used"]
            summary["train_real_size"] = info["train_real_size"]
            summary["valid_size"] = info["valid_size"]
            summary["test_size"] = info["test_size"]
            summary["syn_total_used"] = info["syn_total_used"]
            summary["train_final_size"] = info["train_final_size"]

            all_runs.append(summary)

            pd.DataFrame([summary]).to_csv(
                os.path.join(combo_dir, "results_summary.csv"),
                index=False,
                encoding="utf-8-sig"
            )

            print(f"[Saved] {combo_dir}/epoch_metrics.csv")
            print(f"[Saved] {combo_dir}/classification_report_best.txt")
            print(f"[Saved] {combo_dir}/per_class_scores_best.csv")
            print(f"[Saved] {combo_dir}/results_summary.csv")
            print(f"Best epoch={summary['best_epoch']}, test_F1={summary['test_f1']:.4f}, ckpt={summary['best_checkpoint']}")

# 全體彙總
df_all = pd.DataFrame(all_runs).sort_values(
    by=["dlr_ratio", "syn_ratio", "sample_seed", "test_f1", "test_accuracy"],
    ascending=[True, True, True, False, False]
).reset_index(drop=True)

df_all.to_csv(os.path.join(OUT_ROOT, "all_runs_summary.csv"), index=False, encoding="utf-8-sig")

print(f"\n[Saved] {OUT_ROOT}/all_runs_summary.csv")
print(df_all.head(20))

Mounted at /content/drive
transformers version: 5.0.0
Device: cpu
No GPU detected, running on CPU
Loaded DLR real data: 7434 posts
Loaded SYN data: 10281 posts

===== Building datasets | DLR=50% | SYN=30% | seed=11 =====


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]